In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

## 1. Feature Transformation

In [2]:
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
data.drop(["RowNumber", "CustomerId", "Surname"], axis=1, inplace=True)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Geography and Gender is categorical

#### Label Encoding for Gender

In [4]:
label_encoder_gender = LabelEncoder()

data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [5]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [6]:
data["Geography"].unique()

array(['France', 'Spain', 'Germany'], dtype=object)

#### One Hot for Geography since more than 2 values

In [7]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder_geo = OneHotEncoder()

geo_encoder = onehot_encoder_geo.fit_transform(data[["Geography"]])

geo_encoder

<10000x3 sparse matrix of type '<class 'numpy.float64'>'
	with 10000 stored elements in Compressed Sparse Row format>

In [8]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [9]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [10]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


### Combine with original data

In [11]:
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


### Save the encoders and scaler

In [12]:
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

### Divide the Dataset into independent and dependent features

In [13]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [14]:
X = data.drop('Exited', axis=1)
y = data['Exited']

#### Split into Train Test

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

#### Scale these Features

In [16]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#### Save as pickle

In [17]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

## 2. ANN Implementation

In [29]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import Input
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

### Build ANN Model

In [32]:
(X_train.shape[1],) # how many columns will be input
X_train.shape[1]


12

In [30]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu'), # HL1 connected with input layer
    Dense(32, activation='relu'), ## HL2
    Dense(1, activation='sigmoid') ## output layer
]
)

In [33]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 64)                  │             832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

### Compile the model
- in order to do forward and back propogation we need to compile the model

#### Using this method we can define our own learning rate

In [35]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
# opt = tf.keras.optimizers.Adadelta(learning_rate=0.01)
# opt = tf.keras.optimizers.Adamax(learning_rate=0.01)

#### using this method we can use loss

In [36]:
loss = tf.keras.losses.BinaryCrossentropy()
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x000001A364445EE0>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

#### Using this method there will be its own learning rate so we don't define optimizer in string instead we use opt

In [37]:
model.compile(optimizer=opt, loss="binary_crossentropy", metrics=['accuracy'])

### Set up Tensorboard

In [39]:
logs_dir = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [41]:
tensorflow_callback = TensorBoard(log_dir=logs_dir, histogram_freq=1)

### Setup Early Stopping
- we can train model for any epochs for any batch size
- lets say we run for 100 epochs but if in 20 epochs only model is trained well at best level and after that it is just varying by 1 or 2% then it is not necessary to train for 100 epochs
- we have to monitor loss value - if loss value is decreasing then fine but not then apply early stopping and if loss value is not decreasing then stop
- patience means be patient for atleast 5 epochs if till the 5 epochs there is no improvement then you can stop it over there
- when going thru forward and back propogation at which epoch we find out the best weight, you can consider that and you can reload it when you stop the early stopping

In [42]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [44]:
history = model.fit(X_train, 
                    y_train, 
                    validation_data=(X_test, y_test), 
                    epochs=100,
                    callbacks = [tensorflow_callback, early_stopping_callback]
                   )

Epoch 1/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.8023 - loss: 0.4499 - val_accuracy: 0.8612 - val_loss: 0.3643
Epoch 2/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8379 - loss: 0.3728 - val_accuracy: 0.8670 - val_loss: 0.3403
Epoch 3/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8551 - loss: 0.3549 - val_accuracy: 0.8579 - val_loss: 0.3544
Epoch 4/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8569 - loss: 0.3457 - val_accuracy: 0.8639 - val_loss: 0.3316
Epoch 5/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8638 - loss: 0.3343 - val_accuracy: 0.8664 - val_loss: 0.3327
Epoch 6/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8526 - loss: 0.3466 - val_accuracy: 0.8582 - val_loss: 0.3448
Epoch 7/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8642 - loss: 0.3353 - val_accuracy: 0.8670 - val_loss: 0.3314
Epoch 8/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8673 - loss: 0.3295 - va

### Save the model

In [46]:
model.save('model.keras')

### Load Tensorboard Extension

In [48]:
%load_ext tensorboard

In [53]:
logs_dir
from pathlib import Path

Path(logs_dir)

WindowsPath('logs/fit20250326-023940')

In [55]:
# %tensorboard --logdir logs/fit20250326-023940

## 3. Prediction

In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

### Load all trained model, scaler pickle, onehot

In [18]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)

model = load_model('model.keras')
model.compile(optimizer=opt, loss="binary_crossentropy", metrics=['accuracy'])

C:\Users\gurunaml\OneDrive - Firstsource Solutions Ltd\Desktop\ML\ML\Lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [5]:
with open('onehot_encoder_geo.pkl', 'rb') as file:
    one_hot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [4]:
# Example input

input_data = {
    'CreditScore': 600,
    "Geography": 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

#### one hot encode 'Geography'

In [6]:
geo_encoded = one_hot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns= one_hot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

C:\Users\gurunaml\OneDrive - Firstsource Solutions Ltd\Desktop\ML\ML\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


#### Convert Dict to DF

In [7]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


#### combine one hot encoded columns with input data

In [8]:
input_df = pd.concat([input_df.reset_index(drop=True), geo_encoded_df], axis=1)
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


#### Encode categorical variables

In [9]:
input_df['Gender'] = label_encoder_gender.transform(input_df["Gender"])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


#### drop Geography Column

In [10]:
input_df.drop("Geography", inplace=True, axis=1)

In [11]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


#### Scaling the input data

In [12]:
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53912042,  0.9089063 ,  0.10586484, -0.68234612, -0.25753129,
         0.81978044,  0.64675311,  0.96566376, -0.88398762,  0.9961269 ,
        -0.57183516, -0.57987798]])


#### Predict Churn

In [21]:
prediction = model.predict(input_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step


In [22]:
prediction

array([[0.04210508]], dtype=float32)

#### Get Prediction Probability

In [15]:
prediction_proba = prediction[0][0]
prediction_proba

0.042105082

In [16]:
if prediction_proba > 0.5:
    print("The customer is likely to churn")

else:
    print('The customer is not likely to churn')

The customer is not likely to churn
